# Lab 7.1 — A measured SampleNTT validity distinguisher

**ChipWhisperer Husky + CW308_STM32F4 · variable-time A0 only**

Can a power trace predict whether **the first 12-bit candidate is accepted** (`d0 < 3329`) for a seed the classifier has never seen? This version replaces the early-window toy classifier. The previous 64-seed experiment was inconclusive (balanced accuracy ≈ 0.52) and is preserved as historical data.

We first record the complete operation, locate the first candidate pair with a **separate calibration image**, then restore the ordinary sampler. We collect balanced development seeds with the same total workload, choose a window using development data, freeze the classifier, and only then capture fresh final-test seeds. Confidence intervals, a shuffled-label control, and a timing-only baseline decide what we can claim. **A negative result is a valid outcome.**

All acquisition cells below operate your physical board. There is no simulated-data fallback. Calibration firmware instruments call boundaries; neither training nor evaluation accepts its traces. FLS is deferred.

**Start with a fresh kernel** using `chipwhisperer-new/.venv/bin/python`. Close the old notebook's device connection before restarting. Run the numbered sections in order, inspecting the plots. Do not use Run All for the first hardware pass. Old captures are never overwritten.

## 1. Load the lab

The companion directory stays under `jupyter/courses/custom`. The upstream sampler remains pinned and hash checked. Installing dependencies is a separate environment step; this cell does not open USB.

In [1]:
from pathlib import Path
import sys, time, json
from datetime import datetime, timezone
from uuid import uuid4
here = Path.cwd().resolve()
roots = [here, *list(here.parents)[:6]]
candidates = [p / tail for p in roots for tail in (
    'mlkem-samplentt-power-lab', 'custom/mlkem-samplentt-power-lab',
    'courses/custom/mlkem-samplentt-power-lab',
    'jupyter/courses/custom/mlkem-samplentt-power-lab')]
if (here / 'analysis/distinguisher.py').is_file(): candidates.insert(0, here)
LAB = next((p for p in candidates if (p/'analysis/distinguisher.py').is_file()), None)
if LAB is None: raise FileNotFoundError('Set LAB to the companion folder under jupyter/courses/custom.')
sys.path.insert(0, str(LAB))
CW_ROOT = next(p for p in LAB.parents if (p/'firmware/mcu/Makefile.inc').is_file())
import numpy as np
import matplotlib.pyplot as plt
import chipwhisperer as cw
from IPython.display import display, Markdown
from tqdm.auto import tqdm
# A notebook may retain older module objects after files change on disk.
# Refresh the lab code without reconnecting the existing scope/target.
import importlib
for module_name in ('analysis.distinguisher', 'analysis.distinguisher_plots',
                    'capture.hardware', 'capture.cohort', 'firmware.build'):
    module = importlib.import_module(module_name)
    if not Path(module.__file__).resolve().is_relative_to(LAB.resolve()):
        raise RuntimeError(f'{module_name} is cached from another lab folder. Restart the kernel and select this checkout.')
    importlib.reload(module)
from analysis.ground_truth import ground_truth, generated_seed
from analysis.c_reference import verify_sources
from analysis.distinguisher import (make_plan, load_plan, window_grid, develop, freeze,
                                    load_frozen, evaluate, cohort_arrays)
from analysis.distinguisher_plots import survey_plot, development_plot, evaluation_plot
from capture.hardware import HardwareBackend, HardwareSettings
from capture.records import save_dataset, load_dataset
from capture.cohort import acquire_cohort, read_cohort
from firmware.build import build
print('Python:', sys.executable)
print('ChipWhisperer:', cw.__version__, 'Repository:', CW_ROOT)
print('Pinned sampler:', verify_sources()['commit'])

Python: /home/ubuntu/Desktop/work/chipwhisperer-new/.venv/bin/python
ChipWhisperer: 7.0.0 Repository: /home/ubuntu/Desktop/work/chipwhisperer-new
Pinned sampler: fd72834bd789c644b52fe0d62b2fa3b9f34dfaec


## 2. Configure this experiment

Default: 128 seeds **per class** in development and another 128 per class in the final test, two captures per seed (1,024 short traces total). The primary result uses repeat 0 only; the mean of two captures is secondary. Reduce the counts only before making a plan. Smaller samples give wider uncertainty.

Both classes have **exactly 160 candidate pairs / three SHAKE blocks**, and candidate 1 is accepted. The final candidate fills the output: exactly 256 acceptances and 64 rejections per seed. This controls the most obvious total-workload and adjacent-candidate confounds. It creates a deliberately conditioned seed population; it does not estimate performance on ordinary random ML-KEM seeds. Seeds are known public inputs in this profiling lab; predicting their labels does not recover a secret.

To resume an interrupted run, set `RESUME_RUN` to its printed folder before running this cell. Never reuse a final test for model tuning.

In [2]:
SCOPE_SN = None  # Set the Husky serial only if more than one device is attached.
PLATFORM = 'CW308_STM32F4'
TARGET_FREQ = 7_372_800
UART_BAUD = 38_400
GAIN_DB = 10.0  # Previously usable gain; clipping still stops capture.
PER_CLASS = 128
REPEATS = 2
PAIR_ITERATIONS = 160
X_INDEX, Y_INDEX = 0, 0
RESUME_RUN = None  # e.g. LAB/'results/runs/vt-v2-20261028T...'
if RESUME_RUN is None:
    RUN = LAB/'results/runs'/('vt-v2-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid4().hex[:6])
    RUN.mkdir(parents=True)
else:
    RUN = Path(RESUME_RUN).expanduser().resolve()
    assert RUN.is_dir(), 'Resume folder does not exist'
print('Keep this path to resume:', RUN)

Keep this path to resume: /home/ubuntu/Desktop/work/chipwhisperer-new/jupyter/courses/custom/mlkem-samplentt-power-lab/results/runs/vt-v2-20261028T213312Z-d99cd4


## 3. Connect and define capture helpers

Use the target board's normal Husky clock/UART wiring and TIO4 trigger. These helpers retain this connection; they do not open another USB session. Any changed gain, clock, firmware, or window requires a fresh experiment after development capture has begun.

In [3]:
scope = cw.scope(sn=SCOPE_SN)
if 'husky' not in str(scope).lower():
    scope.dis()
    raise RuntimeError('This notebook requires Husky.')
scope.default_setup()
scope.clock.clkgen_freq = TARGET_FREQ
scope.clock.adc_mul = 4
scope.io.hs2 = 'clkgen'
scope.io.tio1, scope.io.tio2, scope.io.tio3 = 'serial_rx', 'serial_tx', 'high_z'
scope.trigger.module = 'basic'
scope.trigger.triggers = 'tio4'
target = cw.target(scope, cw.targets.SimpleSerial)
target.baud = UART_BAUD

def reset_target():
    scope.io.nrst = 'low'; time.sleep(.05)
    scope.io.nrst = 'high_z'; time.sleep(.25)
    target.flush()

def configure_window(multiplier, samples, offset):
    scope.clock.adc_mul = multiplier
    scope.adc.stream_mode = False
    scope.adc.presamples = 0
    scope.adc.segments = 1
    scope.adc.decimate = 1
    scope.adc.samples = int(samples)
    scope.adc.offset = int(offset)
    scope.adc.basic_mode = 'rising_edge'
    scope.adc.timeout = 10
    scope.gain.db = GAIN_DB
    scope.adc.errors = 0
    time.sleep(.25)
    assert scope.clock.adc_locked, 'ADC clock did not lock'

def attach_recorder(build_path):
    settings = HardwareSettings(model='Husky', target_platform=PLATFORM,
        build_json=str(build_path), scope_serial=str(scope.sn),
        target_clock_hz=float(scope.clock.clkgen_freq), adc_multiplier=int(scope.clock.adc_mul),
        samples=int(scope.adc.samples), offset=int(scope.adc.offset), gain_db=float(scope.gain.db),
        baud=int(target.baud), timeout_seconds=float(scope.adc.timeout),
        stream_mode=bool(scope.adc.stream_mode))
    return HardwareBackend.from_connected(scope,target,settings,cw_version=cw.__version__)

print('Connected:', scope.sn)
print('Actual target clock:', scope.clock.clkgen_freq, 'Hz')

(ChipWhisperer Scope ERROR|File OpenADC.py:425) Unknown hwInfoVer: Default/Unknown


ValueError: Unknown ChipWhisperer: 

## 4. Build and program the representative sampler

The trigger still encloses the complete upstream A0 (SHAKE, candidate loop and allocation). UART, output hashing and coefficient initialization lie outside it. This build contains no calibration wrapper. Programming is explicit because the identity handshake checks variant/revision/profile, rather than reading back a complete binary hash.

In [ ]:
# Reload here too: rerunning this cell must not call a cached old builder.
import importlib
builder_module = importlib.import_module('firmware.build')
if Path(builder_module.__file__).resolve() != (LAB/'firmware/build.py').resolve():
    raise RuntimeError('The cached firmware builder belongs to another lab folder; restart the kernel.')
build = importlib.reload(builder_module).build
measurement_build = build(CW_ROOT, PLATFORM, 'variable_time', optimization='2', profile='measurement')
MEASUREMENT_JSON = LAB/'firmware/variable_time/build.json'
print('Image:', measurement_build['image'])
print('SHA256:', measurement_build['firmware_sha256'])

Image: /home/ubuntu/Desktop/work/chipwhisperer-new/jupyter/courses/custom/mlkem-samplentt-power-lab/firmware/variable_time/samplentt-variable_time-CW308_STM32F4.hex
SHA256: fbe72dbef701cfd52eb08b3ca895c66112c9296e3d89797c0ff2301b318f91f3


In [ ]:
# Check metadata before programming, so an old manifest gives a useful error.
manifest = json.loads(MEASUREMENT_JSON.read_text())
if manifest.get('profile') != 'measurement':
    raise RuntimeError('Missing/wrong firmware profile. Rerun the preceding build cell to reload the builder and regenerate build.json.')
cw.program_target(scope, cw.programmers.STM32FProgrammer, measurement_build['image'])
reset_target()
SURVEY_SAMPLES = min(130_000, int(scope.adc.max_samples))
configure_window(1, SURVEY_SAMPLES, 0)
recorder = attach_recorder(MEASUREMENT_JSON)
print('Verified representative profile:', recorder.build['profile'])

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 15203 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 15203 bytes


(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:704) Target clock may drop; you may need to reset your target.


Verified representative profile: measurement


## 5. Capture the complete operation

This low-rate survey must contain the entire trigger. Plotting is decimated for display only. Expect the first Keccak work before the first rejection test; the old 0–4,000 sample ROI cannot establish candidate leakage.

If ADC clipping occurs, lower `GAIN_DB`, reconfigure and repeat before creating the plan. Do not disable clipping checks. If the full operation exceeds buffer capacity, stop and configure a slower target or a separate streaming survey before continuing.

In [4]:
survey_records = []
for i in tqdm(range(4), desc='Full-operation survey'):
    r = recorder.capture(generated_seed(i, 'vt-v2-survey'), X_INDEX, Y_INDEX)
    if not r.metadata['full_operation_captured']:
        raise RuntimeError('Survey does not cover the complete operation; fix its duration before proceeding.')
    survey_records.append(r)
    print(i, r.truth['pair_iterations'], 'pairs;', round(r.metadata['duration_seconds']*1e3,3), 'ms')
save_dataset(RUN/('survey-'+uuid4().hex[:6]),survey_records,[],{'purpose':'measurement survey'})
fig = survey_plot(survey_records[0]); display(fig); plt.close(fig)
raw = survey_records[0].wave
print('Raw ADC range:', float(raw.min()), float(raw.max()), '; resolution:', survey_records[0].metadata['adc_bits'])

Full-operation survey:   0%|          | 0/4 [00:00<?, ?it/s]

NameError: name 'recorder' is not defined

## 6. Locate the first pair with a separate calibration build

The calibration image uses the Cortex-M4 cycle counter and a linker wrapper around `shake128_inc_squeeze`. It records entry/return of the first two calls; the **first candidate pair is processed between the first return and the second entry**. There is no serial I/O in the trigger. The vendored sampler source is unchanged.

These timestamps are approximate guides: wrapper overhead and flash layout differ from the representative image. We retain generous padding and check the ordinary waveform afterward. Calibration traces are excluded by the analysis code. Identity bit 7 distinguishes this image from the ordinary one.

In [ ]:
# Reload here too: rerunning this cell must not call a cached old builder.
import importlib
builder_module = importlib.import_module('firmware.build')
if Path(builder_module.__file__).resolve() != (LAB/'firmware/build.py').resolve():
    raise RuntimeError('The cached firmware builder belongs to another lab folder; restart the kernel.')
build = importlib.reload(builder_module).build
calibration_build = build(CW_ROOT, PLATFORM, 'variable_time', optimization='2', profile='calibration')
CALIBRATION_JSON = LAB/'firmware/variable_time_calibration/build.json'
print('Calibration image:', calibration_build['image'])

In [ ]:
manifest = json.loads(CALIBRATION_JSON.read_text())
if manifest.get('profile') != 'calibration':
    raise RuntimeError('Missing/wrong calibration profile. Rerun the preceding calibration build cell.')
cw.program_target(scope, cw.programmers.STM32FProgrammer, calibration_build['image'])
reset_target()
configure_window(1, SURVEY_SAMPLES, 0)
calibrator = attach_recorder(CALIBRATION_JSON)
print('Verified profile:', calibrator.build['profile'])

In [ ]:
calibration_records, markers = [], []
for i in tqdm(range(8), desc='Locate first pair'):
    r = calibrator.capture(generated_seed(i, 'vt-v2-calibration'), X_INDEX, Y_INDEX)
    mark = calibrator.calibration_timing()
    core_cycles_from_trigger = r.metadata['duration_seconds']*float(scope.clock.clkgen_freq)
    if abs(mark['total_cycles']-core_cycles_from_trigger) > max(256,.02*core_cycles_from_trigger):
        raise RuntimeError('DWT and trigger clock scales disagree; inspect the target clock before locating the ROI')
    if mark['squeeze_calls'] != r.truth['pair_iterations']:
        raise RuntimeError('Calibration call count disagrees with independent SHAKE schedule')
    r.metadata['calibration_cycles'] = mark
    calibration_records.append(r); markers.append(mark)
save_dataset(RUN/('calibration-'+uuid4().hex[:6]),calibration_records,[],{'purpose':'localization only'})
CORE_HZ = float(scope.clock.clkgen_freq)
print('First SHAKE return (core cycles):', [m['first_squeeze_return'] for m in markers])
print('First-pair interval (core cycles):', [m['second_squeeze_start']-m['first_squeeze_return'] for m in markers])
# Include 512 core cycles before and at least 1536 after the located pair.
start_cycle = max(0, min(m['first_squeeze_return'] for m in markers)-512)
stop_cycle = max(m['second_squeeze_start'] for m in markers)+1536
ADC_MUL = 4
ROI_OFFSET = int(start_cycle*ADC_MUL)
ROI_SAMPLES = int(np.ceil(max(8192,(stop_cycle-start_cycle)*ADC_MUL)/128)*128)
assert ROI_SAMPLES <= int(scope.adc.max_samples)
localization = dict(core_hz=CORE_HZ, markers=markers, adc_multiplier=ADC_MUL,
                    offset=ROI_OFFSET, samples=ROI_SAMPLES, gain_db=GAIN_DB,
                    caveat='Approximate calibration boundaries; instrumented layout and wrapper add overhead.')
# A resumed experiment must restore its original window, not silently replace it.
loc_path = RUN/'localization.json'
if loc_path.exists():
    old = json.loads(loc_path.read_text())
    ROI_OFFSET, ROI_SAMPLES, ADC_MUL, GAIN_DB = old['offset'],old['samples'],old['adc_multiplier'],old['gain_db']
    print('Restoring the saved capture window for this run.')
else:
    loc_path.write_text(json.dumps(localization,indent=2))
print('Representative capture: offset', ROI_OFFSET, 'samples;', ROI_SAMPLES, 'samples at 4× target clock')
fig = survey_plot(survey_records[0],markers[0],CORE_HZ)
display(fig); plt.close(fig)

## 7. Restore the representative image and inspect its located window

**Required before training or testing.** The recorder rejects the calibration identity when attached to this build. We now capture at 4× the target clock with a delayed start. The trigger counter still measures the whole operation even though only a short waveform window is retained. Partial-operation capture is intentional here.

In [ ]:
cw.program_target(scope, cw.programmers.STM32FProgrammer, measurement_build['image'])
reset_target()
configure_window(ADC_MUL, ROI_SAMPLES, ROI_OFFSET)
recorder = attach_recorder(MEASUREMENT_JSON)
located = recorder.capture(generated_seed(99,'vt-v2-check'),X_INDEX,Y_INDEX)
save_dataset(RUN/('located-check-'+uuid4().hex[:6]),[located],[],{'purpose':'representative window check'})
fig, ax = plt.subplots(figsize=(12,3))
t_us = (ROI_OFFSET+np.arange(len(located.wave)))/located.metadata['adc_frequency_hz']*1e6
ax.plot(t_us,located.wave,lw=.6)
for m in markers:
    ax.axvline(m['first_squeeze_return']/CORE_HZ*1e6,color='red',alpha=.2)
ax.set(xlabel='Time since trigger (µs)',ylabel='ADC codes',title='Representative image; red = approximate first SHAKE return')
fig.tight_layout(); display(fig); plt.close(fig)
print('ADC:', located.metadata['adc_frequency_hz'], 'Hz; complete operation:', located.metadata['duration_seconds']*1e3,'ms')

## 8. Declare the seed sets before collecting model data

The plan generates disjoint development and final-test seeds from new domains. It also excludes seeds from old saved runs. Each class has the same exact pair count and accepted second candidate. This generation uses independent Python SHAKE ground truth; the target's polynomial digest verifies every subsequent capture.

The JSON plan is saved once. Reading test labels for tuning invalidates that final test, even though they can be computed from the public seeds.

In [ ]:
PLAN_PATH = RUN/'plan.json'
if PLAN_PATH.exists():
    plan = load_plan(PLAN_PATH)
else:
    excluded = set()
    for manifest_path in (LAB/'results/runs').rglob('manifest.json'):
        if RUN in manifest_path.parents: continue
        saved = json.loads(manifest_path.read_text())
        for stem in saved.get('records',[]):
            metadata_path = manifest_path.parent/(stem+'.json')
            if metadata_path.exists():
                excluded.add(json.loads(metadata_path.read_text())['truth']['seed'])
    plan = make_plan(PLAN_PATH,per_class=PER_CLASS,repeats=REPEATS,
        pair_iterations=PAIR_ITERATIONS,x=X_INDEX,y=Y_INDEX,excluded=excluded)
print('Plan:', plan['sha256'])
print('Seeds per cohort:', len(plan['cohorts']['development']), '; repeats:', plan['repeats'])
print('Workload:', plan['pair_iterations'], 'pairs; candidate 1 always valid')

## 9. Collect development data (resumable)

Labels/repeats are interleaved in a fixed random order. Every success is immediately checkpointed. A capture error stops the batch and records the failed attempt; fix it and rerun this cell to continue. The analysis requires every declared seed/repeat, so failed seeds cannot disappear silently. A change of gain or other recording conditions requires a new run.

You can inspect `attempts.jsonl` for errors or retries. Frequent failures, class-dependent failures, or a hardware reset mid-cohort merit investigation even if a later retry succeeds.

In [ ]:
development_records = acquire_cohort(recorder,plan,'development',RUN/'development',
    progress=lambda items:tqdm(items,desc='Development traces'))
X_dev,y_dev,t_dev,dev_seeds,condition = cohort_arrays(development_records,plan,'development')
print('Complete development cohort:',len(dev_seeds),'independent seeds;',np.bincount(y_dev),'[reject, accept]')
print('Condition:', json.dumps(condition,indent=2))

## 10. Select the power window using development only

We search short windows around the localized first pair. Within each of four cross-validation folds, a Welch score selects at most 16 spaced sample positions **using only that fold's training seeds**. Those points feed scaling and a regularized logistic classifier. No features are selected from the final test.

The best score after searching many windows is optimistic. Treat this plot and CV score as development diagnostics, not the final result. Waveform alignment is fixed by the hardware trigger; there is no label-dependent alignment.

In [ ]:
if (RUN/'frozen-model.json').exists():
    frozen_model = load_frozen(RUN/'frozen-model.json')
    selection = frozen_model['selection']
    print('Using the already frozen selection; no new search.')
else:
    windows = window_grid(ROI_SAMPLES, widths=(128,256,512,1024),step=128)
    selection = develop(development_records,plan,windows)
    (RUN/'development-selection.json').write_text(json.dumps(selection,indent=2))
print('Selected relative window:',selection['best_window'])
print('Best DEVELOPMENT CV BA:', max(s['mean_cv_ba'] for s in selection['search']))
fig = development_plot(development_records,plan,selection)
display(fig); plt.close(fig)

## 11. Freeze the classifier and controls

This stores the selected sample positions, scales, coefficients, threshold, firmware/settings signature, analysis-source hash and a fingerprint of the development data. The artifact uses JSON numeric arrays, with no pickle. It cannot be overwritten by this cell.

The timing-only model is linear logistic regression on total trigger duration, a benchmark rather than an optimal timing attack. A separate model trains on shuffled development labels as a diagnostic control. Both are frozen now. The single-trace power classifier is the predeclared primary endpoint; two-trace averaging is secondary.

In [ ]:
FROZEN_PATH = RUN/'frozen-model.json'
frozen_model = load_frozen(FROZEN_PATH) if FROZEN_PATH.exists() else freeze(FROZEN_PATH,development_records,plan,selection)
print('Frozen SHA256:', frozen_model['sha256'])
print('Relative ROI:', frozen_model['roi'], '; decision threshold:', frozen_model['decision_threshold'])
fig = development_plot(development_records,plan,selection,frozen_model)
fig.savefig(RUN/'development-poi.png',dpi=160)
display(fig); plt.close(fig)

## 12. Acquire the fresh final test

Only run this after freezing. Acquisition refuses to proceed without the frozen artifact and stamps its hash into every test record. Seeds have not appeared in development or old saved runs. Keep all settings and the target image fixed. Do not inspect these data and return to window/model selection.

In [ ]:
holdout_records = acquire_cohort(recorder,plan,'holdout',RUN/'holdout',frozen_path=FROZEN_PATH,
    progress=lambda items:tqdm(items,desc='Fresh final-test traces'))
print('Final-test captures:',len(holdout_records))

## 13. Evaluate once, with uncertainty and controls

Balanced accuracy is the average of reject recall and accept recall; chance is 0.5. ROC AUC measures ranking independent of the fixed threshold. Confidence intervals resample independent seeds, not repeated traces. The one-sided permutation test asks how often shuffled **test labels** would produce at least this balanced accuracy with the already fixed predictions.

The power-minus-timing AUC interval is paired on the same seeds. If it overlaps zero, there is no demonstrated improvement in ranking over total timing. A significant power association still does not identify the physical branch responsible. Shuffled-label performance can fluctuate above chance; unusually strong control performance needs investigation.

Rerunning this cell only recomputes the same fixed evaluation. It is not permission to tune the model against this test.

In [ ]:
frozen_model = load_frozen(FROZEN_PATH)
report = evaluate(holdout_records,plan,frozen_model,bootstrap=2000,permutations=5000)
report['acquisition_audit'] = {}
for role in ('development','holdout'):
    attempts = [json.loads(line) for line in (RUN/role/'attempts.jsonl').read_text().splitlines()]
    failed = [item for item in attempts if item['status']!='ok']
    report['acquisition_audit'][role] = dict(total_attempts=len(attempts), failed_attempts=failed,
        failures_by_label={str(label):sum(int(ground_truth(bytes.fromhex(item['seed']),plan['x'],plan['y']).valid[0])==label for item in failed) for label in (0,1)})
(RUN/'final-report.json').write_text(json.dumps(report,indent=2))
if any(a['failed_attempts'] for a in report['acquisition_audit'].values()):
    print('Recorded capture failures/retries: inspect acquisition_audit before drawing conclusions.')
for mode in ('single_trace','mean_repeats'):
    print('\n'+mode+(' — PRIMARY' if mode=='single_trace' else ' — secondary'))
    for name,result in report['results'][mode].items():
        print(name, 'BA=',round(result['balanced_accuracy'],3),
              '95% CI=',np.round(result['balanced_accuracy_95ci'],3),
              'AUC=',round(result['roc_auc'],3))
primary = report['results']['single_trace']['power']
print('\nPower permutation p:',primary['one_sided_permutation_p_ba'])
print('Power minus timing AUC 95% CI:',primary['auc_minus_timing_95ci'])
print(report['decision'])
fig = evaluation_plot(report); fig.savefig(RUN/'final-evaluation.png',dpi=160)
display(fig); plt.close(fig)

## 14. Interpret and save the conclusion

Look at the **primary single-trace** result first:

- BA near 0.5, or an interval including 0.5: the experiment has not demonstrated a convincing single-trace distinguisher. Preserve that negative result.
- BA's lower confidence bound above 0.5 and a small permutation p-value: evidence of a validity association for these independent seeds under these conditions. Replicate with a fresh run before making a stronger claim.
- Two-trace averaging alone works: report a two-capture result, not a single-trace result.
- Power and timing perform similarly: do not claim that power adds discrimination beyond timing.
- Shuffled labels work unusually well, frequent retries occur, or the settings changed: investigate before interpreting the result.

This lab classifies a known property of a public SHAKE input in a balanced, workload-conditioned population. It does not demonstrate secret-key recovery, causal isolation of the rejection branch, natural-prior precision, or generalization to another target/build. Any next model iteration requires a **new plan and a fresh final test**.

In [ ]:
primary = report['results']['single_trace']['power']
conclusion = f"""# SampleNTT validity experiment

{report['decision']}

Independent final-test seeds: {report['n_seeds']}.
Primary single-trace balanced accuracy: {primary['balanced_accuracy']:.3f}, 95% CI {primary['balanced_accuracy_95ci']}.
ROC AUC: {primary['roc_auc']:.3f}; permutation p (balanced accuracy): {primary['one_sided_permutation_p_ba']:.5f}.
Power minus timing AUC 95% CI: {primary['auc_minus_timing_95ci']}.

Plan: `{plan['sha256']}`
Frozen fit: `{frozen_model['sha256']}`

"""+'\n'.join('- '+item for item in report['limits'])
(RUN/'conclusion.md').write_text(conclusion)
display(Markdown(conclusion))
for role in ('development','holdout'):
    attempts = [json.loads(s) for s in (RUN/role/'attempts.jsonl').read_text().splitlines()]
    failed = [a for a in attempts if a['status']!='ok']
    print(role, 'recorded failed attempts:',len(failed))
print('All results and checkpoints:',RUN)

## 15. Reload saved results without hardware

Optional after a kernel restart: run sections 1–2 with `RESUME_RUN` set, then this cell. No device connection is needed to check the frozen model and complete saved cohorts. Never replace a missing cohort with synthetic data.

In [ ]:
plan = load_plan(RUN/'plan.json')
frozen_model = load_frozen(RUN/'frozen-model.json')
development_records = read_cohort(RUN/'development')
holdout_records = read_cohort(RUN/'holdout')
report = json.loads((RUN/'final-report.json').read_text())
cohort_arrays(development_records,plan,'development')
cohort_arrays(holdout_records,plan,'holdout')
print('Saved decision:', report['decision'])
fig = evaluation_plot(report); display(fig); plt.close(fig)

## 16. Disconnect when finished

This releases the device for another notebook or process. There is no Husky firmware update in this lab.

In [ ]:
target.dis()
scope.dis()
print('Disconnected.')